# 03 · 문서를 AI Search에 넣기

## 학습 목표
- 문서 → 청크 → 임베딩 → 인덱스 → 업로드 단계를 구분합니다.
- 출처 메타데이터와 1536차원 벡터 필드를 확인합니다.

## 개념
임베딩은 텍스트를 유사도 검색용 숫자 벡터로 바꿉니다. 검색 인덱스에 넣는 것과 LLM을 학습시키는 것은 다릅니다. **검색 가능하게 저장하는 작업**입니다.

`data/policies/`의 문서는 이 실습용으로 새로 작성한 가상 한빛테크 규정입니다. 실제 사내 문서·개인정보를 넣지 마세요. 출처는 파일명·절·청크 ID를 보존합니다. Markdown의 `page=1`은 논리 페이지이며 실제 PDF 쪽수가 아닙니다.

⚠️ 이 노트북은 **본인 인덱스를 생성/갱신하고 문서를 업로드**합니다. 개인 키의 서버 측 권한은 본인 인덱스로 제한됩니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


In [ ]:
from hanbit_assistant.ingestion import load_policy_chunks, chunk_markdown
chunks = load_policy_chunks(ROOT / "data/policies")
print("문서 수:", len({c["source"] for c in chunks}), "청크 수:", len(chunks))
print(json.dumps(chunks[2], ensure_ascii=False, indent=2))
assert len(chunks) > 0


### 1. 청크 크기를 바꿔 보기
기본 청킹은 제목/절을 우선 보존합니다. 샘플 절이 짧으므로 아래에서는 일부러 작은 크기를 써서 분할과 겹침을 관찰합니다. 작은 청크가 항상 더 좋은 것은 아닙니다.

In [ ]:
text = (ROOT / "data/policies/02_lodging.md").read_text(encoding="utf-8")
small_chunks = chunk_markdown(text, "data/policies/02_lodging.md", max_chars=100, overlap=15)
print("기본:", len([c for c in chunks if c["source"].endswith("02_lodging.md")]))
print("작은 청크:", len(small_chunks))
for part in small_chunks[-3:]:
    print(part["section"], len(part["content"]), part["content"])


### 2. 임베딩과 인덱스 스키마
한 쿼리 벡터와 문서 벡터는 **같은 모델·차원**을 사용해야 합니다. HNSW는 가까운 벡터를 빠르게 찾기 위한 알고리즘입니다. 인덱스에는 한국어 키워드 검색 필드와 벡터 필드를 함께 둡니다.

In [ ]:
from hanbit_assistant.search import build_index, embed_texts, PolicySearch
async with create_openai_client(settings) as client:
    example_vectors = await embed_texts(client, ["도쿄 호텔 비용 한도", "일본 출장 숙박비"])
print("실제 벡터 차원:", [len(v) for v in example_vectors])
print("첫 벡터 앞 5개 값:", example_vectors[0][:5])
print([(f.name, f.type) for f in build_index(settings.search_index).fields])
assert all(len(v) == 1536 for v in example_vectors)


### 3. 생성·업로드·실제 읽기 검증
리소스(Search 서비스)는 강사가 미리 만듭니다. 학생은 서비스 안의 **자기 인덱스**만 다룹니다. `create_index()`는 인덱스 구조, `upload_chunks()`는 임베딩 생성과 문서 업로드입니다.

In [ ]:
import asyncio
service = PolicySearch(settings)
try:
    await service.create_index()
    uploaded = await service.upload_chunks(chunks)
    for attempt in range(30):
        count = await service.client.get_document_count()
        if count >= len(chunks):
            break
        await asyncio.sleep(1)
    assert count >= len(chunks), "검색 반영이 지연됩니다. 잠시 후 개수를 재확인하세요."
    print({"uploaded": uploaded, "indexed_count": count, "index": settings.search_index})
    hits = await service.search("도쿄 숙박비", mode="keyword", top=3)
    print([(h["title"], h["section"]) for h in hits])
finally:
    await service.close()


## 관찰
같은 원문을 재실행하면 동일 ID로 upsert되어 중복이 늘지 않습니다. 그러나 텍스트가 바뀌면 ID도 바뀝니다. **수정·삭제한 문서의 옛 청크를 제거하는 전체 동기화 기능은 아닙니다.** 기본 실습에서는 원문을 수정하지 않습니다. 실제 운영은 버전 관리·삭제 동기화가 추가로 필요합니다.

## 연습
출처 필드가 없으면 답변의 어느 부분을 검증할 수 없게 될까요? 1536차원 인덱스에 다른 차원의 벡터를 보내면 어떻게 될까요?

<details><summary>예시 해설</summary>출처가 없으면 검색된 문장의 원문·절·버전을 찾기 어렵습니다. 차원 불일치는 업로드/검색 오류가 납니다. 모델 이름만 바꾸지 말고 임베딩 모델과 인덱스 차원·기존 데이터까지 함께 관리해야 합니다.</details>